In [ ]:
!pip install -q unsloth
!pip install -q --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
!pip install -U unsloth unsloth_zoo

In [ ]:
import torch, transformers, trl, unsloth
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("unsloth:", unsloth.__version__)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3047,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
from datasets import load_dataset
dataset = load_dataset('ServiceNow-AI/R1-Distill-SFT', 'v0', split = "train")

In [ ]:
print(dataset[:5])

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

EOS_TOKEN = tokenizer.eos_token

In [ ]:
def formatting_prompts_func(examples):
  problems = examples["problem"]
  thoughts = examples["reannotated_assistant_content"]
  solutions = examples["solution"]
  texts = []

  for problem, thought, solution in zip(problems, thoughts, solutions):
    messages = [
        {"role": "system", "content" : "You are a reflective assistant engaging in thorough, iterative reasoning, mimicking human stream-of-consciousness thinking. Your approach emphasizes exploration, self-doubt, and continuous refinement before coming up with an answer."},
        {"role": "user", "content" : problem},
        {"role": "assistant", "content": f"{thought}\n\n{solution}"},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt = False,
    ) + EOS_TOKEN

    texts.append(text)

  return {"text": texts}

In [ ]:
dataset = dataset.map(formatting_prompts_func, batched = True,)

In [ ]:
print(dataset[0])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3047,
        output_dir = "outputs",
        report_to = "none",
        average_tokens_across_devices = False,
    ),
  )

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a reflective assistant engaging in thorough, iterative reasoning, mimicking human stream-of-consciousness thinking. Your approach emphasizes exploration, self-doubt, and continuous refinement before coming up with an answer."
    },
    {
        "role": "user",
        "content": "How many 'r's are present in 'strawberry'?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)

inputs = {k: v.to("cuda") for k, v in inputs.items()}

with torch.inference_mode():
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=512,
        do_sample=False,
        use_cache=False,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.batch_decode(
    outputs[:, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print(response[0])

In [ ]:
# FastLanguageModel.for_inference(model)

# messages = [
#     {"role": "system", "content" : "You are a reflective assistant engaging in thorough, iterative reasoning, mimicking human stream-of-consciousness thinking. Your approach emphasizes exploration, self-doubt, and continuous refinement before coming up with an answer."},
#     {"role": "user", "content" : "How many 'r's are present in 'strawberry'?"}
# ]

# inputs = tokenizer.apply_chat_template(
#     messages,
#     tokenize = True,
#     add_generation_prompt = True,
#     return_tensors = "pt",
# ).to("cuda")

In [ ]:
# print(inputs)

In [ ]:
# outputs = model.generate(**inputs, max_new_tokens = 1024, use_cache = True, temperature = 1.5, min_p = 0.1)

In [ ]:
# print(outputs[0])

In [ ]:
# response = tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[1]:], skip_special_tokens = True,)

In [ ]:
# print(response[0])

In [ ]:
# model.save_pretrained("kunal-qwen2.5-3b-instruct-001")
# tokenizer.save_pretrained("kunal-qwen2.5-3b-instruct-001")

In [ ]:
# model.save_pretrained_gguf("kunal-qwen2.5-3b-instruct-001-GGUF", tokenizer,)

In [ ]:
# model.save_pretrained_merged("kunal-qwen2.5-3b-instruct-001-merged", tokenizer,)

In [ ]:
# model.push_to_hub_merged("kgaero/kunal-qwen2.5-3b-instruct-001-merged", tokenizer, token = "YOUR_HF_TOKEN")

In [ ]:
# !rm -rf kunal-qwen2.5-3b-instruct-001-merged

In [ ]:
# !rm -rf kunal-qwen2.5-3b-instruct-001-GGUF

In [ ]:
# !rm -rf kunal-qwen2.5-3b-instruct-001

In [ ]:
# !rm -rf ~/.cache/huggingface
# !rm -rf ~/.cache/pip

In [ ]:
model.push_to_hub_gguf(
    "kgaero/kunal-qwen2.5-3b-instruct-001",
    tokenizer,
    quantization_method = ["q4_k_m"],
    token = "YOUR_HF_TOKEN"
)

In [ ]:
model.save_pretrained_gguf(
    "kunal-qwen2.5-3b-instruct-001",
    tokenizer,
    quantization_method = ["q4_k_m"]
)

In [ ]:
!ls kunal-qwen2.5-3b-instruct-001*

!ls kunal-qwen2.5-3b-instruct-001/*
!ls kunal-qwen2.5-3b-instruct-001_gguf/*

In [ ]:
!apt-get update
!apt-get install -y zstd

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time
proc = subprocess.Popen(["ollama", "serve"])
time.sleep(10)
!ollama list

In [ ]:
!cat kunal-qwen2.5-3b-instruct-001_gguf/Modelfile

In [ ]:
!ollama create unsloth-model -f kunal-qwen2.5-3b-instruct-001_gguf/Modelfile

In [ ]:
# print(tokenizer._ollama_modelfile)

In [ ]:
# modelfile_path = "kunal-qwen2.5-3b-instruct-001/Modelfile"

# with open(modelfile_path, "w") as f:
#     f.write(tokenizer._ollama_modelfile)

# print("Modelfile written to", modelfile_path)

In [ ]:
# import os
# os.chdir('kunal-qwen2.5-3b-instruct-001')
# !ollama create unsloth-model -f Modelfile

In [ ]:
# !ollama create unsloth-model -f ./kunal-qwen2.5-3b-instruct-001/Modelfile

In [ ]:
!ollama run unsloth-model "How many r's are in strawberry?"

In [ ]:
!ollama run unsloth-model "A bat and a ball cost $1.10 in total. The bat costs $1 more than the ball. How much does the ball cost? Explain carefully."

In [ ]:
!ollama list